<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/agentic_agent_student_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [1]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 5.2 MB/s eta 0:00:00


## 1) Define KB

In [2]:
import json

# Cette base de connaissances sert de mémoire sémantique locale pour l'agent.
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
    {'source': 'kb:smolagents', 'text': 'Smolagents is a lightweight library by Hugging Face for building tool-calling agents.'},
    {'source': 'kb:loop', 'text': 'The loop consists of observation, thought, action, and feedback steps.'}
]

print('KB entries:', len(kb_snippets))

KB entries: 7


## 2) Define tools

In [4]:
from smolagents import Tool, TransformersModel, ToolCallingAgent
import json

class KBLookupTool(Tool):
    name = "kb_lookup"
    description = "Recherche des informations dans la base de connaissances interne."
    inputs = {"query": {"type": "string", "description": "Le terme ou la phrase à rechercher."}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"\n{item['text']}\n[{item['source']}]\n"
            for item in self.kb
            if any(word in item["text"].lower() for word in q.split())
        ]
        return "".join(matches) if matches else "No matching knowledge found."

class MathTool(Tool):
    name = "math_tool"
    description = "Effectue des additions ou des multiplications."
    inputs = {
        "a": {"type": "number", "description": "Premier nombre"},
        "b": {"type": "number", "description": "Deuxième nombre"},
        "op": {"type": "string", "description": "L'opération : 'add' ou 'multiply'", "nullable": True}
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "add":
            res = a + b
        elif op == "multiply":
            res = a * b
        else:
            return "Error: Unknown operation."
        return json.dumps({"result": res})

kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

## 3) Model (tiny local)

In [6]:
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Utilisation d'un modèle Instruct plus performant pour le tool-calling
model = TransformersModel(
    model_id=MODEL_ID,
    max_new_tokens=200,
    temperature=0.1
)

print("Nouveau modèle prêt :", MODEL_ID)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Nouveau modèle prêt : HuggingFaceTB/SmolLM2-135M-Instruct


## 4) Agent

In [7]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=3,
    instructions="You are a senior AI agent. Use tools to answer. Be concise (2-4 sentences). If no info, say you can't find it.",
)

print("Agent initialized with tools: KBLookup and MathTool")

Agent initialized with tools: KBLookup and MathTool


## 5) Test queries

In [8]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("\n" + "="*20)
    print("Question:", q)
    # L'agent analyse la question, choisit l'outil via le modèle et l'exécute.
    result = agent.run(q)
    print("Final answer:", result)


Question: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - HuggingFaceTB/SmolLM2-135M-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Error while parsing tool call from model output: The model output does not contain any JSON blob.

...

Error while parsing tool call from model output: The model output does not contain any JSON blob.

[Step 3: Duration 8.32 seconds| Input tokens: 4,183 | Output tokens: 600]

Reached max steps.

[Step 4: Duration 8.39 seconds| Input tokens: 5,055 | Output tokens: 800]

Final answer: An agentic AI loop is a type of AI loop that involves repeatedly asking the AI to perform a task, and the AI is repeatedly asked to perform the same task, until it reaches a state where it can perform the task without any further loops.

In other words, an agentic AI loop is a loop that involves repeatedly asking the AI to perform a task, and the AI is repeatedly asked to perform the same task, until it reaches a state where it can perform the task without any further loops.

For example, imagine an AI system that is asked to perform a task, such as "What is the capital of France?" and then asked to perform the same task, such as "What is the capital of France?" and then asked to perform the same task, such as "What is the capital of France?" and then asked to perform the same task, such as "What is the capital of France?" and then asked to perform the same task, such as "What
